# ResumeScanner — Kaggle Capstone Demo Notebook

> **Kaggle AI Agents: Intensive Vibe Coding Capstone Project** (Agents for Business)
> 
> This notebook demonstrates the full multi-agent pipeline end-to-end.
> It runs **without Docker, without a web server, and without API keys**
> (the ML classifier and security tools are fully local/offline).

## What this notebook shows

1. **SkillScanFile** — magic-byte MIME validation (security scanner)
2. **SkillRedactPII** — regex PII detection and redaction
3. **SkillScoreResume** — local ML classification (TF-IDF + classifier, v6)
4. **SkillGenerateFeedback** — FeedbackAgent LLM reasoning (Gemini/Groq or deterministic fallback)
5. **Full multi-agent pipeline** — all 4 steps chained end-to-end
6. **Audit trail** — what gets written to `audit_log` after every step

## Course concepts demonstrated

| Concept | Where |
|---------|-------|
| Agent / Multi-agent system (ADK) | Cells 5–8: SecurityOrchestratorAgent + FeedbackAgent |
| MCP Server | Cell 9: mcp_server.py tool list |
| Security features | Cells 2–4: scan_file, redact_pii, audit_log |
| Agent skills (Agents CLI) | Cells 5–7: SKILL_REGISTRY invocation |
| Deployability | Docker Compose section at bottom |

## Setup — add backend to path

Run this from the repository root (or adjust the path as needed).

In [ ]:
import sys, os

# Adjust this path if running from a different working directory
BACKEND_PATH = os.path.join(os.getcwd(), 'FullStackApp', 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

# Load .env for optional API keys (Gemini / Groq)
# The notebook works without keys — the ML pipeline is fully offline.
try:
    from dotenv import load_dotenv
    env_path = os.path.join(BACKEND_PATH, '.env')
    if os.path.exists(env_path):
        load_dotenv(env_path)
        print('Loaded .env')
    else:
        print('No .env found — LLM features will use deterministic fallback')
except ImportError:
    print('python-dotenv not installed — set GEMINI_API_KEY manually if needed')

print('Backend path:', BACKEND_PATH)
print('Python:', sys.version)

## 1. SkillScanFile — Magic-byte MIME validation

In [ ]:
from app.tools.security_scanner import scan_file
import json

# Test 1: Valid PDF magic bytes
fake_pdf = b'%PDF-1.4 test content for demo'
result = scan_file(fake_pdf, 'resume.pdf')
print('Valid PDF bytes:')
print(json.dumps(result, indent=2))

print()

# Test 2: Windows PE executable (MZ header) renamed to .pdf — MUST be rejected
fake_exe = b'MZ\x90\x00\x03\x00\x00\x00' + b'\x00' * 100
result2 = scan_file(fake_exe, 'resume.pdf')
print('Executable bytes renamed to .pdf:')
print(json.dumps(result2, indent=2))
assert result2['passed'] is False, 'Security scan should have rejected this!'
print('\n✅ Security scan correctly rejected the executable file.')

## 2. SkillRedactPII — PII detection and redaction

In [ ]:
from app.tools.pii_redactor import redact_pii

sample_resume = """
John Doe
Email: john.doe@gmail.com
Phone: +91-9876543210
PAN: ABCDE1234F

Software Engineer with 5 years of Python and ML experience.
Skills: Python, FastAPI, scikit-learn, PostgreSQL, Docker
"""

result = redact_pii(sample_resume)

print(f"PII items redacted: {result['redaction_count']}")
print(f"Types found: {result['types_found']}")
print()
print('Redacted text (safe for LLM):')  
print(result['redacted_text'])
print()
print('Original text unchanged (for DB/recruiter UI):')
print(sample_resume[:80] + '...')

## 3. Agent Skills Registry — discover all 5 skills

In [ ]:
from app.agents.agent_skills import SKILL_REGISTRY

skills = SKILL_REGISTRY.list_skills()
print(f'Registered skills: {len(skills)}\n')
for skill in skills:
    llm_tag = ' [LLM]' if skill.requires_llm else ''
    db_tag  = ' [DB]'  if skill.requires_db  else ''
    print(f'  {skill.name} ({skill.category}){llm_tag}{db_tag}')
    print(f'    {skill.description[:80]}...')
    print()

# Verify MCP export format
mcp_tools = SKILL_REGISTRY.to_mcp_tool_list()
print(f'MCP tools exported: {len(mcp_tools)}')
for t in mcp_tools:
    print(f'  {t["name"]}: {list(t["inputSchema"]["properties"].keys())}')

## 4. SkillScoreResume — Local ML classification (offline, no API key needed)

In [ ]:
result = SKILL_REGISTRY.invoke('SkillScoreResume', {'resume_text': sample_resume})

print('ML Classification result:')
print(f"  Predicted category: {result.get('predicted_category')}")
print(f"  Confidence:         {result.get('confidence', 0):.1%}")
if result.get('top_categories'):
    print('  Top categories:')
    for cat in result['top_categories'][:3]:
        if isinstance(cat, dict):
            print(f"    {cat.get('category')}: {cat.get('confidence', 0):.1%}")
if result.get('error'):
    print(f"  Note: {result['error']}")

## 5. SkillGenerateFeedback — FeedbackAgent (Agent 2)

Uses Gemini → Groq → deterministic fallback. Works without API keys.

In [ ]:
redact_result = redact_pii(sample_resume)
score_result  = SKILL_REGISTRY.invoke('SkillScoreResume', {'resume_text': sample_resume})

feedback = SKILL_REGISTRY.invoke('SkillGenerateFeedback', {
    'redacted_resume_text': redact_result['redacted_text'],
    'score_result':         score_result,
    'job_description':      None,   # no JD — uses predicted category for gap analysis
})

print('FeedbackAgent output:')
print(f"  LLM used:      {feedback.get('agent_used_llm')}")
print(f"  Category fit:  {feedback.get('category_fit')}")
print(f"  ATS summary:   {feedback.get('ats_summary')}")
print()
print('Top skill gaps:')
for gap in feedback.get('skill_gaps', []):
    print(f'  • {gap}')
print()
print('Priority improvements:')
for imp in feedback.get('improvements', []):
    impact = imp.get('impact', '')
    boost  = imp.get('ats_score_boost', 0)
    action = imp.get('action', '')
    print(f'  [{impact}] +{boost} ATS pts: {action[:70]}...' if len(action) > 70 else f'  [{impact}] +{boost} ATS pts: {action}')

## 6. Full Multi-Agent Pipeline (end-to-end)

This mirrors exactly what `python cli_agent.py run-pipeline resume.pdf` does.

In [ ]:
import time

print('=' * 60)
print('  MULTI-AGENT PIPELINE DEMO')
print('=' * 60)

t0 = time.perf_counter()

# AGENT 1 — SecurityOrchestratorAgent
print('\nAGENT 1 — SecurityOrchestratorAgent')
print('  Role: security validation + PII protection + ML classification')

# Step 1: Scan
fake_pdf_bytes = b'%PDF-1.4 test resume content'
scan = SKILL_REGISTRY.invoke('SkillScanFile', {
    'file_path': BACKEND_PATH + '/../model.pkl',  # use any real file for scan demo
    'filename': 'resume.pkl',
})
# For the demo, just test with bytes directly
from app.tools.security_scanner import scan_file as _sf
scan = _sf(fake_pdf_bytes, 'resume.pdf')
print(f"\n  Step 1 — SkillScanFile: passed={scan['passed']} type={scan['detected_type']}")

# Step 2: Redact PII
redact = SKILL_REGISTRY.invoke('SkillRedactPII', {'text': sample_resume})
print(f"  Step 2 — SkillRedactPII: {redact['redaction_count']} PII item(s) redacted ({', '.join(redact['types_found']) or 'none'})")
print(f"           PII types: {redact['types_found']}")
print(f"           LLM payloads will use redacted text — PII stays local.")

# Step 3: Score resume
score = SKILL_REGISTRY.invoke('SkillScoreResume', {'resume_text': sample_resume})
print(f"  Step 3 — SkillScoreResume: {score.get('predicted_category')} ({score.get('confidence', 0):.1%} confidence)")

# AGENT 2 — FeedbackAgent
print('\nAGENT 2 — FeedbackAgent')
print('  Role: LLM-driven resume feedback + improvement prioritization')
feedback = SKILL_REGISTRY.invoke('SkillGenerateFeedback', {
    'redacted_resume_text': redact['redacted_text'],
    'score_result':         score,
})
print(f"  Step 4 — SkillGenerateFeedback: LLM used={feedback.get('agent_used_llm')}, fit={feedback.get('category_fit')}")

# Step 5: Audit log (console mode — no DB)
SKILL_REGISTRY.invoke('SkillLogAudit', {
    'step_name': 'demo_pipeline',
    'status':    'passed',
    'detail':    f"Demo notebook: {redact['redaction_count']} PII items, category={score.get('predicted_category')}",
    'resume_id': None,
})

elapsed = time.perf_counter() - t0
print('\n' + '=' * 60)
print('  Pipeline Complete')
print('=' * 60)
print(f"  Security scan:  {'PASSED' if scan['passed'] else 'FAILED'}")
print(f"  PII redacted:   {redact['redaction_count']} item(s) — {', '.join(redact['types_found']) or 'none'}")
print(f"  ML category:    {score.get('predicted_category')} ({score.get('confidence', 0):.1%})")
print(f"  Category fit:   {feedback.get('category_fit')}")
print(f"  Total time:     {elapsed:.2f}s")

## 7. MCP Server — list tools

Shows the 5 tools the MCP server exposes to any MCP-aware client (Claude Desktop, ADK agents).

In [ ]:
from app.mcp_server import MCP_TOOLS

print(f'MCP tools registered: {len(MCP_TOOLS)}\n')
for tool in MCP_TOOLS:
    params = list(tool['inputSchema']['properties'].keys())
    required = tool['inputSchema'].get('required', [])
    print(f'  Tool: {tool["name"]}')
    print(f'    {tool["description"][:80]}...')
    print(f'    Params: {params}  (required: {required})')
    print()

print('To run the MCP server interactively:')
print('  cd FullStackApp/backend')
print('  python -m app.mcp_server')
print()
print('To inspect with MCP Inspector:')
print('  npx @modelcontextprotocol/inspector python -m app.mcp_server')

## 8. Deployability — Docker Compose

The app is fully containerised with a single command. This satisfies the **Deployability** course concept.

```bash
# From the repository root
docker-compose up --build
```

| Service | URL |
|---------|-----|
| React Frontend | http://localhost:5173 |
| FastAPI Backend | http://localhost:8000 |
| Swagger Docs | http://localhost:8000/docs |
| Audit Trail API | http://localhost:8000/api/audit-log |

The Docker Compose file (`docker-compose.yml`) orchestrates:
- **postgres** — PostgreSQL 15 with health check and named volume
- **backend** — FastAPI (waits for DB health check before starting)
- **frontend** — Vite React dev server with hot-module replacement

## Summary — Course Concepts Demonstrated

| Concept | Demonstrated | Evidence |
|---------|-------------|----------|
| Agent / Multi-agent (ADK) | ✅ | Cells 5–6: SecurityOrchestratorAgent + FeedbackAgent |
| MCP Server | ✅ | Cell 7: 5 tools with inputSchema |
| Security features | ✅ | Cells 1–2: magic-byte scan, PII redaction |
| Agent skills (CLI) | ✅ | Cells 3–6: SKILL_REGISTRY with 5 self-describing skills |
| Deployability | ✅ | Cell 8: Docker Compose one-command deploy |
| Antigravity | 📹 | Shown in video submission |

**5 of 6 key concepts demonstrated in code. All 6 with the video.**